# Demonstrasi API hbsaemp di Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madsyair/hbsaemp/blob/main/demo/demo_api_colab.ipynb)

Notebook ini menjalankan alur yang sama dengan `demo/demo_api.py`: pemanggilan API *hbsaemp*
secara berurutan mulai dari pembacaan data hingga estimasi area (alur Gambar 11). Karena
*hbsaemp* belum dipublikasikan di PyPI, pustaka dipasang langsung dari repositori GitHub
[madsyair/hbsaemp](https://github.com/madsyair/hbsaemp).

| Langkah | Tahap | Fungsi |
|---|---|---|
| 2 | Memuat dataset | `load_dataset()` |
| 3 | Spesifikasi model (tingkat pemula) | `hbm_beta()` |
| 4 | Pengecekan prior | `check_prior()` |
| 6 | Pemasangan model | `model.fit()` |
| 7 | Diagnostik konvergensi | `check_convergence()` |
| 10 | Perbandingan dengan model `create_model` | `create_model()` + `compare_models()` |
| 14 | Estimasi area | `estimate_areas()` |
| 15 | Hasil estimasi area | `result_table` |

Dataset: `data_betalogitnorm` (30 area, mengikuti dataset hbsaems di R). Model: Beta
logit-normal dengan presisi dari desain survei (`n`, `deff`).

Jalankan dengan **Runtime → Run all** agar durasi di akhir notebook mencerminkan waktu
komputasi. Kedua pemodelan MCMC membutuhkan beberapa menit. Bagian terakhir (opsional)
menjalankan GUI *hbsaemp* di Colab.

## Persiapan: instalasi dari GitHub

*hbsaemp* membutuhkan Python 3.12 atau lebih baru. Sel pertama memeriksa versi Python runtime
Colab, sel kedua memasang *hbsaemp* dari branch `main` dengan *extra* `[gui]`, yang mencakup
*backend* Bayesian (Bambi, PyMC, ArviZ) dan dependensi GUI (Panel, dcor, seaborn).

In [ ]:
import sys

print(sys.version)
assert sys.version_info >= (3, 12), (
    "hbsaemp membutuhkan Python >= 3.12. Ganti runtime Colab ke versi Python yang lebih baru."
)

In [ ]:
%pip install -q "hbsaemp[gui] @ git+https://github.com/madsyair/hbsaemp.git@main"

Jika sel berikutnya gagal mengimpor karena paket yang baru diperbarui (misalnya galat
versi NumPy), pilih **Runtime → Restart session**, lalu lanjutkan dari sel impor di bawah
tanpa menjalankan ulang sel instalasi.

In [ ]:
from datetime import datetime

import hbsaemp as hb

waktu_mulai = datetime.now()
print("Versi hbsaemp:", hb.__version__)

# cfg_beta untuk config model beta (tingkat pemula)
# cfg_cm untuk config model create_model (tingkat menengah)
cfg_beta = hb.ModelConfig(draws=2000, tune=1000, chains=4, target_accept=0.95, random_seed=42)
cfg_cm = hb.ModelConfig(draws=3000, tune=1500, chains=6, target_accept=0.95, random_seed=42)

## Langkah 2: Memuat dataset

In [ ]:
df = hb.load_dataset("data_betalogitnorm")
print(f"Ukuran data: {df.shape[0]} area x {df.shape[1]} kolom")
df.head()

## Langkah 3: Spesifikasi model melalui `hbm_beta` (tingkat pemula)

In [ ]:
model_beta = hb.hbm_beta(
    response="y",
    auxiliary=["x1", "x2", "x3"],
    data=df,
    n="n",
    deff="deff",
    area_var="group",
    config=cfg_beta,
)
print("Formula yang dirakit:", model_beta.formula)
print(model_beta.summary())

## Langkah 4: Pengecekan prior

Prior diperiksa sebelum model dipasang, sehingga model masih berstatus belum dipasang
(`is_fitted = False`).

In [ ]:
prior = hb.check_prior(model_beta)
print(prior.summary())
print("Model sudah dipasang?", model_beta.is_fitted)
prior.prior_summary

## Langkah 6: Pemasangan model (`fit`)

In [ ]:
model_beta.fit()
print(model_beta.summary())

## Langkah 7: Diagnostik konvergensi

In [ ]:
conv = hb.check_convergence(model_beta, plot_types=["trace", "rhat"])
print(conv.summary())
conv.rhat_ess[["mean", "sd", "r_hat", "ess_bulk", "ess_tail"]]

## Langkah 10: Perbandingan dengan model yang dibangun melalui `create_model`

Model pembanding dispesifikasikan dengan formula lengkap (tingkat menengah) dan memakai
konfigurasi sampler yang berbeda. Peringatan Pareto *k* dari ArviZ dapat muncul pada model
tingkat area karena setiap area hanya memiliki satu observasi.

In [ ]:
model_cm = hb.create_model(
    "y ~ x1 + x2 + (1|group)",
    family="beta",
    data=df,
    n="n",
    deff="deff",
    config=cfg_cm,
)
print(model_cm.summary())
model_cm.fit()
print(hb.check_convergence(model_cm, plot_types=[]).summary())

In [ ]:
cmp = hb.compare_models([model_beta, model_cm])
print(cmp.summary())
cmp.comparison_table

In [ ]:
kandidat = {"model_0": model_beta, "model_1": model_cm}
nama_terbaik = cmp.comparison_table.index[0]
model_terbaik = kandidat[nama_terbaik]
print(f"Model terbaik menurut ELPD-LOO: {nama_terbaik} ({model_terbaik.formula})")

## Langkah 14: Estimasi area

In [ ]:
est = hb.estimate_areas(model_terbaik)
print(est.summary())

## Langkah 15: Hasil estimasi area

Tabel lengkap disimpan sebagai `langkah15_estimasi_area.csv` di folder kerja Colab
(`/content`) dan dapat diunduh melalui panel **Files**.

In [ ]:
hasil = est.result_table.merge(df[["group", "y", "theta"]], on="group")
hasil = hasil.rename(columns={"y": "penduga_langsung", "theta": "nilai_sebenarnya"})
hasil.to_csv("langkah15_estimasi_area.csv", index=False)
hasil[["group", "penduga_langsung", "mean", "ci_lower", "ci_upper",
       "rse_pct", "nilai_sebenarnya"]].round(4)

## Waktu eksekusi

In [ ]:
waktu_selesai = datetime.now()
durasi = (waktu_selesai - waktu_mulai).total_seconds()
print(f"Waktu mulai   : {waktu_mulai:%Y-%m-%d %H:%M:%S}")
print(f"Waktu selesai : {waktu_selesai:%Y-%m-%d %H:%M:%S}")
print(f"Durasi        : {durasi:.1f} detik ({durasi / 60:.1f} menit)")

## Opsional: menjalankan GUI *hbsaemp* di Colab

Di laptop, GUI cukup dijalankan dengan `hb.launch_app()` atau perintah `hbsaemp-app`. Cara itu
tidak dapat dipakai di Colab karena dua hal:

- port bawaan GUI, 8080, sudah dipakai oleh Colab sendiri, sehingga muncul galat
  `OSError: [Errno 98] Address already in use`;
- `launch_app()` menahan sel dan mencoba membuka browser di mesin Colab.

Karena itu, server GUI dijalankan di latar belakang pada port yang kosong, lalu dibuka melalui
proxy Colab. Setelah sel berikut dijalankan, klik tautan yang muncul untuk membuka GUI di tab
baru. Grafik GUI dirender di server (`Agg`), jadi setelah bagian ini dijalankan, grafik sel lain
tidak lagi tampil langsung di notebook sampai sesi di-*restart*.

In [ ]:
import socket

import matplotlib

matplotlib.use("Agg")  # grafik GUI dirender di server, sama seperti launch_app()

import panel as pn
from google.colab import output
from hbsaemp.app import App, AppConfig

# Pilih port yang sedang kosong (port 8080 dipakai oleh Colab).
with socket.socket() as s:
    s.bind(("", 0))
    PORT_GUI = s.getsockname()[1]

app = App(app_config=AppConfig(port=PORT_GUI, open_browser=False))
server = pn.serve(app.view, port=PORT_GUI, show=False, threaded=True,
                  allow_websocket_origin=["*"])
output.serve_kernel_port_as_window(PORT_GUI)

Jalankan sel berikut untuk menghentikan server GUI. `allow_websocket_origin=["*"]` membuka
akses dari alamat mana pun, jadi hanya gunakan untuk sesi Colab sementara.

In [ ]:
server.stop()
print("Server GUI dihentikan.")